# Predictive Anayltics: Support Vector Machines with Classification

TODO: add embedded, check why there are Nans in lanlong

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [1]:
DO_GRID_SEARCH = True
GRID_SAMPLE = 10_000
SPATIAL_UNIT = "HEXAGON" # options: CENSUS_TRACTS, HEXAGON, COMMUNITY_AREAS
SPATIAL_ENCODING = "latlong" # options: embedding, latlong, onehot
MODE = "full" # options: full, sample
TIME_UNIT = "4H" # options: 1H, 2H, 4H
H3_RES = "8" # options 7,8

In [2]:
CENSUS_PATH = "../data/full/raw_data/Census_Tracts.csv"
COMM_PATH = "../data/full/raw_data/Community_Areas.csv"

In [3]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib as plt
import datetime
from sklearn.svm import SVC 
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from joblib import load, dump
import h3

from shapely import wkt

import geopandas as gpd
from shapely.geometry import Polygon
from srai.neighbourhoods import H3Neighbourhood
from srai.loaders import OSMOnlineLoader
from srai.regionalizers import H3Regionalizer, geocode_to_region_gdf
from srai.joiners import IntersectionJoiner
from srai.h3 import ring_buffer_h3_regions_gdf
from srai.embedders import Hex2VecEmbedder
import pytorch_lightning as py_light

from sklearn.preprocessing import OneHotEncoder

import networkx as nx
from libpysal.weights import Queen
from node2vec import Node2Vec

C:\Users\bkran\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
##TODO: Combination that are not possible




# check spatial unit default: community area
# check combination
# ist Grid_seach > num(val) setzt auf num(val)

## Preparations

In [5]:
INPUT = "../data/" + MODE + "/train_test_data/" 

In [ ]:
# Paths, depending on spatial and time unit
DATA_PATH_TRAIN = INPUT + "GOLD_" + TIME_UNIT + "_DEMAND_" + SPATIAL_UNIT + "_TRAIN.parquet"
DATA_PATH_TEST = INPUT + "GOLD_" + TIME_UNIT + "_DEMAND_" + SPATIAL_UNIT + "_TEST.parquet"
DATA_PATH_VAL = INPUT + "GOLD_" + TIME_UNIT + "_DEMAND_" + SPATIAL_UNIT + "_VAL.parquet"
if SPATIAL_UNIT == "HEXAGON":
    DATA_PATH_TEST = INPUT + "GOLD_" + TIME_UNIT + "_DEMAND_" + SPATIAL_UNIT + "_" + H3_RES + "_VAL.parquet"
    DATA_PATH_TRAIN = INPUT + "GOLD_" + TIME_UNIT + "_DEMAND_" + SPATIAL_UNIT + "_" + H3_RES + "_VAL.parquet"
    DATA_PATH_VAL = INPUT + "GOLD_" + TIME_UNIT + "_DEMAND_" + SPATIAL_UNIT + "_" + H3_RES + "_VAL.parquet"



MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_demand"
EXCLUDE_COLS = [
    TARGET_COL, # gets encoded
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "trip_count",
    "date",
    "h3_cell", # spatial units are getting encoded
    "census_tract",
    "community_area",
    "lat",
    "lon"    
]

Load data and select features and target

In [7]:
# Load data
train = pd.read_parquet(DATA_PATH_TRAIN)
test = pd.read_parquet(DATA_PATH_TEST)
val = pd.read_parquet(DATA_PATH_VAL)

In [8]:
print(train.info())

<class 'pandas.DataFrame'>
RangeIndex: 373614 entries, 0 to 373613
Data columns (total 58 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   datetime_hour             373614 non-null  datetime64[us]
 1   month                     373614 non-null  int8          
 2   weekday                   373614 non-null  int8          
 3   hour                      373614 non-null  int8          
 4   month_sin                 373614 non-null  float64       
 5   month_cos                 373614 non-null  float64       
 6   weekday_sin               373614 non-null  float64       
 7   weekday_cos               373614 non-null  float64       
 8   hour_sin                  373614 non-null  float64       
 9   hour_cos                  373614 non-null  float64       
 10  tmpc                      373614 non-null  float64       
 11  relh                      373614 non-null  float64       
 12  sknt         

In [9]:
## TODO: brauchen wir das überhaupt?
keep_cols = [
        "trip_count",  # target is derived from this
        "month_sin", "month_cos", "weekday_sin", "weekday_cos",
        "hour_sin", "hour_cos",
        "tmpc", "relh", "sknt", "vsby", "p01m",
        "skyc1_BKN", "skyc1_CLR", "skyc1_FEW", "skyc1_OVC", "skyc1_SCT", "skyc1_VV ",
        "is_holiday",
        "food_drink", "landmark", "shop", "train_station",
]

keep_cols = np.array(keep_cols)

if SPATIAL_UNIT == "HEXAGON":
    keep_cols = np.append(keep_cols,"h3_cell")
elif SPATIAL_UNIT == "CENSUS_TRACTS":
    keep_cols = np.append(keep_cols,"community_area")
elif SPATIAL_UNIT == "COMMUNITY_AREAS":
    keep_cols = np.append(keep_cols,"community_area")
    print("here")

In [10]:
print(keep_cols)

['trip_count' 'month_sin' 'month_cos' 'weekday_sin' 'weekday_cos'
 'hour_sin' 'hour_cos' 'tmpc' 'relh' 'sknt' 'vsby' 'p01m' 'skyc1_BKN'
 'skyc1_CLR' 'skyc1_FEW' 'skyc1_OVC' 'skyc1_SCT' 'skyc1_VV ' 'is_holiday'
 'food_drink' 'landmark' 'shop' 'train_station' 'h3_cell']


In [11]:
train_df = train[keep_cols].copy()
test_df = test[keep_cols].copy()
val_df = val[keep_cols].copy()

In [12]:
train_df.isna().sum()

trip_count           0
month_sin            0
month_cos            0
weekday_sin          0
weekday_cos          0
hour_sin             0
hour_cos             0
tmpc                 0
relh                 0
sknt                 0
vsby                 0
p01m                 0
skyc1_BKN            0
skyc1_CLR            0
skyc1_FEW            0
skyc1_OVC            0
skyc1_SCT            0
skyc1_VV             0
is_holiday           0
food_drink       82782
landmark         82782
shop             82782
train_station    82782
h3_cell              0
dtype: int64

In [13]:
train_df.count()

trip_count       373614
month_sin        373614
month_cos        373614
weekday_sin      373614
weekday_cos      373614
hour_sin         373614
hour_cos         373614
tmpc             373614
relh             373614
sknt             373614
vsby             373614
p01m             373614
skyc1_BKN        373614
skyc1_CLR        373614
skyc1_FEW        373614
skyc1_OVC        373614
skyc1_SCT        373614
skyc1_VV         373614
is_holiday       373614
food_drink       290832
landmark         290832
shop             290832
train_station    290832
h3_cell          373614
dtype: int64

In [14]:
train_df["food_drink"] = train_df["food_drink"].fillna(0.0)
train_df["landmark"] = train_df["landmark"].fillna(0.0)
train_df["shop"] = train_df["shop"].fillna(0.0)
train_df["train_station"] = train_df["train_station"].fillna(0.0)

val_df["food_drink"] = val_df["food_drink"].fillna(0.0)
val_df["landmark"] = val_df["landmark"].fillna(0.0)
val_df["shop"] = val_df["shop"].fillna(0.0)
val_df["train_station"] = val_df["train_station"].fillna(0.0)

test_df["food_drink"] = test_df["food_drink"].fillna(0.0)
test_df["landmark"] = test_df["landmark"].fillna(0.0)
test_df["shop"] = test_df["shop"].fillna(0.0)
test_df["train_station"] = test_df["train_station"].fillna(0.0)


In [15]:
train_df["food_drink"]

0          2.0
1          3.0
2          3.0
3          3.0
4          0.0
          ... 
373609     1.0
373610     1.0
373611     0.0
373612     0.0
373613    13.0
Name: food_drink, Length: 373614, dtype: float64

In [16]:
# prepare data
# calculate median to split in low/high demand
# when trip_count above 50 percent use "high", when below or equal to 50 percent low


# definition for demand cause currently the q1 is 0, q2 is 1 and q3 is 3
# I excluded the zeros since a magority of values are zero
train_p75 = np.percentile(train_df.loc[train_df["trip_count"] > 0, "trip_count"], 75)
train_p50 = np.percentile(train_df.loc[train_df["trip_count"] > 0, "trip_count"], 50)
train_p25 = np.percentile(train_df.loc[train_df["trip_count"] > 0, "trip_count"], 25)

train_df["trip_demand"] = 0
train_df.loc[train_df["trip_count"] >= train_p25, "trip_demand"] = 1
train_df.loc[train_df["trip_count"] >= train_p50, "trip_demand"] = 2
train_df.loc[train_df["trip_count"] >= train_p75, "trip_demand"] = 3

val_p75 = np.percentile(val_df.loc[val_df["trip_count"] > 0, "trip_count"], 75)
val_p50 = np.percentile(val_df.loc[val_df["trip_count"] > 0, "trip_count"], 50)
val_p25 = np.percentile(val_df.loc[val_df["trip_count"] > 0, "trip_count"], 25)

val_df["trip_demand"] = 0
val_df.loc[train_df["trip_count"] >= val_p25, "trip_demand"] = 1
val_df.loc[train_df["trip_count"] >= val_p50, "trip_demand"] = 2
val_df.loc[train_df["trip_count"] >= val_p75, "trip_demand"] = 3

test_p75 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 75)
test_p50 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 50)
test_p25 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 25)

test_df["trip_demand"] = 0
test_df.loc[test_df["trip_count"] >= test_p25, "trip_demand"] = 1
test_df.loc[test_df["trip_count"] >= test_p50, "trip_demand"] = 2
test_df.loc[test_df["trip_count"] >= test_p75, "trip_demand"] = 3


In [17]:
# due to substential amount of zeroes zeroes were excluded from calculating 
print(test_df.loc[train_df["trip_count"] == 0, "trip_count"].count())
print(test_df.loc[train_df["trip_count"] > 0, "trip_count"].count())

336371
37243


In [18]:
# Find the size of the smallest class
min_count = train_df['trip_demand'].value_counts().min()

# Sample min_count rows from each class
balanced_df = (
    train_df.groupby('trip_demand', group_keys=False)
    .apply(lambda x: x.sample(min_count, random_state=42))
    .reset_index(drop=False)
)

#train_df = balanced_df

#print(min_count)

In [19]:
balanced_df

,index,trip_count,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,tmpc,relh,...,skyc1_FEW,skyc1_OVC,skyc1_SCT,skyc1_VV,is_holiday,food_drink,landmark,shop,train_station,h3_cell
0,19524,0,8.660254e-01,-5.000000e-01,-0.974928,-0.222521,0.000000e+00,1.0,9.0275,68.8650,...,0,0,0,0,0,0.0,0.0,0.0,0.0,882664ccbdfffff
1,15900,0,1.224647e-16,-1.000000e+00,-0.781831,0.623490,1.224647e-16,-1.0,28.3300,72.6825,...,1,0,0,0,0,2.0,1.0,2.0,0.0,882664cdddfffff
2,61865,0,1.000000e+00,6.123234e-17,-0.974928,-0.222521,-8.660254e-01,0.5,11.2525,51.6725,...,0,0,1,0,0,1.0,0.0,1.0,0.0,882664c883fffff
3,58851,0,5.000000e-01,-8.660254e-01,0.974928,-0.222521,1.224647e-16,-1.0,25.0000,76.1425,...,1,0,0,0,0,0.0,0.0,0.0,0.0,8826641921fffff
4,128363,0,5.000000e-01,8.660254e-01,-0.974928,-0.222521,1.224647e-16,-1.0,-0.1400,73.5075,...,1,0,0,0,0,5.0,2.0,0.0,0.0,882664521bfffff
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37595,239903,29,5.000000e-01,8.660254e-01,-0.781831,0.623490,-8.660254e-01,-0.5,3.4750,52.8375,...,1,0,0,0,0,5.0,1.0,4.0,0.0,882664cf5dfffff
37596,228154,24,1.000000e+00,6.123234e-17,0.000000,1.000000,1.224647e-16,-1.0,7.2200,67.5020,...,0,0,0,0,0,6.0,0.0,1.0,3.0,882664d8cdfffff
37597,232979,88,1.000000e+00,6.123234e-17,0.781831,0.623490,-8.660254e-01,-0.5,2.0825,46.0125,...,1,0,0,0,0,5.0,1.0,4.0,0.0,882664cf5dfffff
37598,76717,78,-8.660254e-01,5.000000e-01,0.974928,-0.222521,1.224647e-16,-1.0,5.6950,61.9475,...,0,0,1,0,0,19.0,1.0,16.0,0.0,882664c1e9fffff


In [20]:
model = SVC()

## Encoding

In [21]:
def feature_cols(train_df):
    feature_cols = [
        col for col in train_df.columns
        if col not in EXCLUDE_COLS
    ]
    return feature_cols


In [22]:
train_df

,trip_count,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,tmpc,relh,sknt,...,skyc1_OVC,skyc1_SCT,skyc1_VV,is_holiday,food_drink,landmark,shop,train_station,h3_cell,trip_demand
0,0,1.000000,6.123234e-17,0.781831,0.623490,0.866025,-0.5,-0.417500,55.720000,11.500000,...,0,0,0,0,2.0,0.0,1.0,0.0,882664d981fffff,0
1,0,-0.500000,-8.660254e-01,0.433884,-0.900969,0.000000,1.0,22.288000,67.930000,9.400000,...,0,1,0,0,3.0,0.0,5.0,2.0,882664cc33fffff,0
2,0,-0.500000,-8.660254e-01,0.433884,-0.900969,-0.866025,0.5,24.442500,58.212500,8.000000,...,0,1,0,0,3.0,0.0,5.0,2.0,882664cc33fffff,0
3,0,1.000000,6.123234e-17,-0.433884,-0.900969,0.866025,0.5,9.784615,98.655385,4.307692,...,1,0,0,0,3.0,0.0,5.0,2.0,882664cc33fffff,0
4,0,0.866025,5.000000e-01,-0.974928,-0.222521,-0.866025,-0.5,18.910000,41.064000,21.400000,...,0,0,0,0,0.0,0.0,0.0,0.0,882664ce2bfffff,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373609,0,0.500000,8.660254e-01,-0.781831,0.623490,0.866025,0.5,-3.984286,78.674286,8.714286,...,1,0,0,0,1.0,2.0,2.0,4.0,882664c8ddfffff,0
373610,0,0.000000,1.000000e+00,0.433884,-0.900969,0.000000,1.0,-6.807500,52.685000,16.750000,...,0,0,0,0,1.0,2.0,2.0,4.0,882664c8ddfffff,0
373611,0,0.866025,5.000000e-01,0.000000,1.000000,0.866025,-0.5,2.500000,60.855000,14.000000,...,1,0,0,0,0.0,0.0,0.0,2.0,882759341bfffff,0
373612,0,0.000000,1.000000e+00,0.433884,-0.900969,0.866025,-0.5,-8.330000,58.632000,16.200000,...,1,0,0,0,0.0,0.0,0.0,2.0,882759341bfffff,0


### Spatial Encoding: LatLong

In [23]:
def spherical_encode(lat, lon):
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    x = np.cos(lat_rad) * np.cos(lon_rad)
    y = np.cos(lat_rad) * np.sin(lon_rad)
    z = np.sin(lat_rad)
    return np.stack([x, y, z], axis=-1)  

In [24]:
# encode into lat long


if (SPATIAL_ENCODING == "latlong"):
    if SPATIAL_UNIT == "HEXAGON":
        print("Encoding: latlong and Unit: hexa")
        for df in (train_df, val_df, test_df):
            df["lat"], df["lon"] = zip(*df["h3_cell"].map(h3.cell_to_latlng))

    elif SPATIAL_UNIT == "CENSUS_TRACTS":
        print("Encoding: latlong and Unit: census_tract")

        # load census tract
        census_data = pd.read_csv(CENSUS_PATH, dtype={"CENSUS_T_1": str})
        census_data["CENSUS_T_1"] = census_data["CENSUS_T_1"].str.zfill(11)

        tract_centroids = census_data.set_index("CENSUS_T_1")[["TRACT_CE_3", "TRACT_CE_2"]]
        tract_centroids.columns = ["lat", "lon"]

        for df in (train_df, test_df):
            df["census_tract"] = df["census_tract"].astype(str).str.zfill(11)
            df["lat"] = df["census_tract"].map(tract_centroids["lat"])
            df["lon"] = df["census_tract"].map(tract_centroids["lon"])

            # sanity check, catch silent join failures early
            n_missing = df["lat"].isna().sum()
            if n_missing:
                print(f"Warning: {n_missing}/{len(df)} rows failed to match a tract centroid")
            n_missing = df["lon"].isna().sum()
            if n_missing:
                print(f"Warning: {n_missing}/{len(df)} rows failed to match a tract centroid")

    elif SPATIAL_UNIT == "COMMUNITY_AREAS":
        print("Encoding: latlong and Unit: community_area")

        census_data = pd.read_csv(COMM_PATH, dtype={"AREA_NUMBE": str})
        census_data["AREA_NUMBE"] = census_data["AREA_NUMBE"].str.zfill(2)

        census_data["geometry"] = census_data["the_geom"].apply(wkt.loads)
        gdf = gpd.GeoDataFrame(census_data, geometry="geometry", crs="EPSG:4326")  

        gdf["lon"] = gdf.geometry.centroid.x
        gdf["lat"] = gdf.geometry.centroid.y

        tract_centroids = gdf.set_index("AREA_NUMBE")[["lat", "lon"]]

        for df in (train_df, test_df):
            df[SPATIAL_UNIT] = df[SPATIAL_UNIT].astype(str).str.zfill(2)
            df["lat"] = df[SPATIAL_UNIT].map(tract_centroids["lat"])
            df["lon"] = df[SPATIAL_UNIT].map(tract_centroids["lon"])

            # sanity check, catch silent join failures early
            n_missing_lat = df["lat"].isna().sum()
            n_missing_lon = df["lon"].isna().sum()
            if n_missing_lat or n_missing_lon:
                print(f"Warning: {n_missing_lat} lat / {n_missing_lon} lon rows failed to match a community area centroid")



Encoding: latlong and Unit: hexa


In [25]:
train_df.head()

,trip_count,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,tmpc,relh,sknt,...,skyc1_VV,is_holiday,food_drink,landmark,shop,train_station,h3_cell,trip_demand,lat,lon
0,0,1.000000,6.123234e-17,0.781831,0.623490,0.866025,-0.5,-0.417500,55.720000,11.500000,...,0,0,2.0,0.0,1.0,0.0,882664d981fffff,0,41.991627,-87.752166
1,0,-0.500000,-8.660254e-01,0.433884,-0.900969,0.000000,1.0,22.288000,67.930000,9.400000,...,0,0,3.0,0.0,5.0,2.0,882664cc33fffff,0,41.753860,-87.623011
2,0,-0.500000,-8.660254e-01,0.433884,-0.900969,-0.866025,0.5,24.442500,58.212500,8.000000,...,0,0,3.0,0.0,5.0,2.0,882664cc33fffff,0,41.753860,-87.623011
3,0,1.000000,6.123234e-17,-0.433884,-0.900969,0.866025,0.5,9.784615,98.655385,4.307692,...,0,0,3.0,0.0,5.0,2.0,882664cc33fffff,0,41.753860,-87.623011
4,0,0.866025,5.000000e-01,-0.974928,-0.222521,-0.866025,-0.5,18.910000,41.064000,21.400000,...,0,0,0.0,0.0,0.0,0.0,882664ce2bfffff,0,41.821467,-87.590449


In [26]:
if SPATIAL_ENCODING == "latlong":
    for df in (train_df, val_df, test_df):
        result = spherical_encode(df["lat"].values, df["lon"].values)  
        df["x"], df["y"], df["z"] = result.T  

    # add x, y, z 
    feature_cols = feature_cols(train_df)

    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]

    val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    X_val_grid = val_df_grid[feature_cols]

### Spatial Encoding: OneHotEncoding

In [27]:
if (SPATIAL_ENCODING == "onehot") & (SPATIAL_UNIT == "COMMUNITY_AREAS"):
    # Community_area is a categorical id, not a numeric quantity, so one-hot encode it
    X_train = pd.get_dummies(train_df[feature_cols], columns=["community_area"])
   # X_val = pd.get_dummies(val_df[feature_cols], columns=["community_area"])
    X_test = pd.get_dummies(test_df[feature_cols], columns=["community_area"])
    # Keep the dummy columns before scaling turns X_train into a plain array
    train_columns = X_train.columns

    # Make sure test has the same dummy columns as train
    X_test = X_test.reindex(columns=train_columns, fill_value=0)
    #X_val = X_val.reindex(columns=train_columns, fill_value=0)

    val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    X_val_grid = pd.get_dummies(val_df_grid[feature_cols], columns=["community_area"])
    X_val_grid = X_val_grid.reindex(columns=train_columns, fill_value=0)


### Spatial Encoding: Spatial Embedding

In [28]:
if (SPATIAL_ENCODING == "embedding") & (SPATIAL_UNIT == "HEXAGON"):

    all_cells = set(train_df["h3_cell"]).union(test_df["h3_cell"])

    regions = []
    for cell in all_cells:
        boundary = h3.cell_to_boundary(cell)
        polygon = Polygon([(lon, lat) for lat, lon in boundary])
        regions.append({"h3_cell": cell, "geometry": polygon})

    regions_gdf = gpd.GeoDataFrame(regions).set_index("h3_cell")

    # Features per H3 cell (mean of POI features, train only)
    poi_cols = ["food_drink", "landmark", "shop", "train_station"]
    features_gdf = train_df.groupby("h3_cell")[poi_cols].mean()

    features_gdf = gpd.GeoDataFrame(
        features_gdf,
        geometry=regions_gdf.loc[features_gdf.index, "geometry"]
    )

    regions_gdf = regions_gdf.rename_axis("region_id")
    features_gdf = features_gdf.rename_axis("feature_id")

    # Required by srai: maps each region to its features
    joiner = IntersectionJoiner()
    joint_gdf = joiner.transform(regions_gdf, features_gdf)

    neighbourhood = H3Neighbourhood(regions_gdf)
    embedder = Hex2VecEmbedder(encoder_sizes=[64, 32])
    embeddings = embedder.fit_transform(
        regions_gdf,
        features_gdf,
        joint_gdf,
        neighbourhood
    )

    emb_cols = [f"emb_{i}" for i in range(embeddings.shape[1])]
    embeddings.columns = emb_cols

    train_df = train_df.merge(embeddings, left_on="h3_cell", right_index=True, how="left")
    test_df = test_df.merge(embeddings, left_on="h3_cell", right_index=True, how="left")

    feature_cols = feature_cols + emb_cols

    X_train = train_df[feature_cols]
    X_val = val_df[feature_cols]
    X_test = test_df[feature_cols]

    val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    X_val_grid = val_df_grid[feature_cols]

elif (SPATIAL_ENCODING == "embedding") & (SPATIAL_UNIT == "CENSUS_TRACTS"):
    census_data = gpd.read_file(CENSUS_PATH)
    census_data["geometry"] = gpd.GeoSeries.from_wkt(census_data["the_geom"])

    gdf = gpd.GeoDataFrame(census_data, geometry="geometry", crs="EPSG:4326")

    gdf["TRACT_FIPS"] = gdf["TRACT_FIPS"].astype(str)
    gdf = gdf.set_index("TRACT_FIPS")

    gdf = gdf.to_crs(epsg=5070)
    gdf = gdf[gdf.geometry.notnull() & gdf.geometry.is_valid]
    gdf = gdf[~gdf.index.duplicated(keep="first")]

    w = Queen.from_dataframe(gdf, use_index=True)
    G = w.to_networkx()
    print(f"Graph: {G.number_of_nodes()} tracts, {G.number_of_edges()} adjacency edges")

    node2vec = Node2Vec(
        G,
        dimensions=32,
        walk_length=20,
        num_walks=100,
        workers=4,
        p=1,
        q=1,
    )

    model = node2vec.fit(window=10, min_count=1, batch_words=4)

    embedding_dict = {node: model.wv[node] for node in G.nodes()}
    emb_df = pd.DataFrame.from_dict(embedding_dict, orient="index")

    emb_cols = [f"emb_{i}" for i in range(emb_df.shape[1])]
    emb_df.columns = emb_cols

    train_df = train_df.merge(emb_df, left_on="census_tract", right_index=True, how="left")
    val_df = val_df.merge(emb_df, left_on="census_tract", right_index=True, how="left")
    test_df = test_df.merge(emb_df, left_on="census_tract", right_index=True, how="left")

    feature_cols = feature_cols + emb_cols

    X_train = train_df[feature_cols]
    X_val = val_df[feature_cols]
    X_test = test_df[feature_cols]

    val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    X_val_grid = val_df_grid[feature_cols]


In [29]:
X_train.head()

,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,tmpc,relh,sknt,vsby,...,skyc1_VV,is_holiday,food_drink,landmark,shop,train_station,lon,x,y,z
0,1.000000,6.123234e-17,0.781831,0.623490,0.866025,-0.5,-0.417500,55.720000,11.500000,10.000000,...,0,0,2.0,0.0,1.0,0.0,-87.752166,0.029151,-0.742671,0.669022
1,-0.500000,-8.660254e-01,0.433884,-0.900969,0.000000,1.0,22.288000,67.930000,9.400000,10.000000,...,0,0,3.0,0.0,5.0,2.0,-87.623011,0.030940,-0.745371,0.665932
2,-0.500000,-8.660254e-01,0.433884,-0.900969,-0.866025,0.5,24.442500,58.212500,8.000000,10.000000,...,0,0,3.0,0.0,5.0,2.0,-87.623011,0.030940,-0.745371,0.665932
3,1.000000,6.123234e-17,-0.433884,-0.900969,0.866025,0.5,9.784615,98.655385,4.307692,0.538462,...,0,0,3.0,0.0,5.0,2.0,-87.623011,0.030940,-0.745371,0.665932
4,0.866025,5.000000e-01,-0.974928,-0.222521,-0.866025,-0.5,18.910000,41.064000,21.400000,6.400000,...,0,0,0.0,0.0,0.0,0.0,-87.590449,0.031331,-0.744567,0.666812


### Create y

In [30]:
y_train = train_df[TARGET_COL]
y_test = test_df[TARGET_COL]
y_val_grid = val_df_grid[TARGET_COL]

In [31]:
feature_cols

['month_sin',
 'month_cos',
 'weekday_sin',
 'weekday_cos',
 'hour_sin',
 'hour_cos',
 'tmpc',
 'relh',
 'sknt',
 'vsby',
 'p01m',
 'skyc1_BKN',
 'skyc1_CLR',
 'skyc1_FEW',
 'skyc1_OVC',
 'skyc1_SCT',
 'skyc1_VV ',
 'is_holiday',
 'food_drink',
 'landmark',
 'shop',
 'train_station',
 'lon',
 'x',
 'y',
 'z']

### Scale


In [32]:
X_train

,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,tmpc,relh,sknt,vsby,...,skyc1_VV,is_holiday,food_drink,landmark,shop,train_station,lon,x,y,z
0,1.000000,6.123234e-17,0.781831,0.623490,0.866025,-0.5,-0.417500,55.720000,11.500000,10.000000,...,0,0,2.0,0.0,1.0,0.0,-87.752166,0.029151,-0.742671,0.669022
1,-0.500000,-8.660254e-01,0.433884,-0.900969,0.000000,1.0,22.288000,67.930000,9.400000,10.000000,...,0,0,3.0,0.0,5.0,2.0,-87.623011,0.030940,-0.745371,0.665932
2,-0.500000,-8.660254e-01,0.433884,-0.900969,-0.866025,0.5,24.442500,58.212500,8.000000,10.000000,...,0,0,3.0,0.0,5.0,2.0,-87.623011,0.030940,-0.745371,0.665932
3,1.000000,6.123234e-17,-0.433884,-0.900969,0.866025,0.5,9.784615,98.655385,4.307692,0.538462,...,0,0,3.0,0.0,5.0,2.0,-87.623011,0.030940,-0.745371,0.665932
4,0.866025,5.000000e-01,-0.974928,-0.222521,-0.866025,-0.5,18.910000,41.064000,21.400000,6.400000,...,0,0,0.0,0.0,0.0,0.0,-87.590449,0.031331,-0.744567,0.666812
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373609,0.500000,8.660254e-01,-0.781831,0.623490,0.866025,0.5,-3.984286,78.674286,8.714286,5.928571,...,0,0,1.0,2.0,2.0,4.0,-87.707453,0.029796,-0.744259,0.667226
373610,0.000000,1.000000e+00,0.433884,-0.900969,0.000000,1.0,-6.807500,52.685000,16.750000,10.000000,...,0,0,1.0,2.0,2.0,4.0,-87.707453,0.029796,-0.744259,0.667226
373611,0.866025,5.000000e-01,0.000000,1.000000,0.866025,-0.5,2.500000,60.855000,14.000000,10.000000,...,0,0,0.0,0.0,0.0,2.0,-87.880086,0.027494,-0.742766,0.668987
373612,0.000000,1.000000e+00,0.433884,-0.900969,0.866025,-0.5,-8.330000,58.632000,16.200000,10.000000,...,0,0,0.0,0.0,0.0,2.0,-87.880086,0.027494,-0.742766,0.668987


In [33]:
# scaling since, SVR is distance-based, so all features need to be on a similar scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_val_grid = scaler.fit_transform(X_val_grid)

In [34]:
X_train

array([[ 1.12639899, -0.32474502,  1.13646904, ..., -0.80818107,
         1.43742256,  1.42943786],
       [-0.93602165, -1.70001657,  0.64010358, ...,  0.73168948,
        -0.802551  , -0.80868965],
       [-0.93602165, -1.70001657,  0.64010358, ...,  0.73168948,
        -0.802551  , -0.80868965],
       ...,
       [ 0.94219101,  0.46926837,  0.02114596, ..., -2.23449626,
         1.35864076,  1.40389145],
       [-0.2485481 ,  1.26328177,  0.64010358, ..., -2.23449626,
         1.35864076,  1.40389145],
       [ 0.43892544,  1.05052652, -1.09417712, ...,  0.13822281,
         0.7172007 ,  0.69513233]])

## Grid Search

In [ ]:
param_grid_linear = {
    "C": [0.1, 1, 10],
    "kernel": ["linear"]
}

param_grid_rbf_sigmoid = {
    "C": [0.1, 1, 10],
    "kernel": ["rbf", "sigmoid"],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

param_grid_poly = {
    "C": [0.1, 1, 10],
    "kernel": ["poly"],
    "degree": [3, 4],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

grids = {}
for name, grid in [("linear", param_grid_linear), ("rbf_sigmoid", param_grid_rbf_sigmoid), ("poly", param_grid_poly)]:
    search = GridSearchCV(
        estimator=SVC(),
        param_grid=grid,
        cv=3,
        scoring="accuracy",
        n_jobs=-1,
        error_score="raise"
    )
    search.fit(X_val_grid, y_val_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

NameError: name 'y_train_grid' is not defined

In [ ]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'C': 10, 'kernel': 'linear'}
Best CV score: 0.6689993234476418


In [ ]:
#missing_mask = test_df["lat"].isna()
#print(test_df.loc[missing_mask, SPATIAL_UNIT].unique())

In [ ]:
# Evaluate on test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

#rint("Test accuracy:", accuracy_score(y_test, y_pred))

## Training Model

In [ ]:
# Train SVC 
best_model.fit(X_train, y_train)

: 

: 

## Testing Model

In [ ]:
# Make prediction 
y_pred = grid_search.predict(X_test)

In [ ]:
# Evaluate Model
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

[[18605   471    96  5921  2521]
 [  959 93086  1727  8517    35]
 [ 1356 30912  1159 12668    23]
 [ 6039 14662   874 24101   129]
 [ 2158     1     1    12 11219]]
              precision    recall  f1-score   support

        High       0.64      0.67      0.66     27614
         Low       0.67      0.89      0.76    104324
         Mid       0.30      0.03      0.05     46118
    Mid High       0.47      0.53      0.50     45805
   Very High       0.81      0.84      0.82     13391

    accuracy                           0.62    237252
   macro avg       0.58      0.59      0.56    237252
weighted avg       0.56      0.62      0.56    237252



## Save Model and Grid Search

In [ ]:
# save model
dump(best_model, "../models/model_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_svc_" + SPATIAL_ENCODING + ".joblib")
dump(grid_search, "../models/grid_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_svc_" + SPATIAL_ENCODING + ".joblib")

['../models/test/grid_community_svc.joblib']